# 01 · Live seismic data, honest freshness

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-WorldView/blob/8e8feea094d489192b8a101069d32fa1328c764f/content/01_live_seismic.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability.

**Provenance:** `content/01_live_seismic.ipynb` at `8e8feea094d489192b8a101069d32fa1328c764f`; original SHA-256 `be2d26ae533a800c143ce212332e75787a782b193ccd6ce9fe4305e7bd98bacb`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 01 · Live seismic data, honest freshness

**Mission:** normalize a public earthquake feed, preserve missing values, and visualize magnitude versus depth. Allow 25 minutes. Prerequisite: Lab 00.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then Run → Run All Cells. First use downloads Matplotlib. All training data are **SYNTHETIC** unless explicitly labeled LIVE. Downloads appear below and in `exports/`; the browser filesystem is separate from your computer. Each lab runs independently.

In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install("matplotlib")
import json, math
from pathlib import Path
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from worldview_lab import (load, export, map_layer, chart_style, haversine_km,
    wilson_interval, shortest_times, freshness, normalize_quakes,
    validate_geojson, brief_request, validate_brief)
chart_style()

## Deterministic first, live on request

Set `USE_LIVE = True` to request the USGS M2.5+ past-day feed. The default bundled fallback is **synthetic**, not a historical USGS snapshot. A failed request prints the reason and retains this label. The feed’s depth is in kilometres and its time is Unix milliseconds. Magnitude is not a measure of local damage.

In [ ]:
USE_LIVE=False
from datetime import datetime, timezone
USGS='https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/2.5_day.geojson'
data=load('seismic_sample.geojson'); provenance='SYNTHETIC fallback (not a USGS observation)'
retrieved_at=None
if USE_LIVE:
    try:
        if sys.platform=='emscripten':
            from pyodide.http import pyfetch
            response=await pyfetch(USGS)
            if not response.ok: raise RuntimeError(f'HTTP {response.status}')
            data=await response.json()
        else:
            import urllib.request
            with urllib.request.urlopen(USGS,timeout=20) as response: data=json.load(response)
        retrieved_at=datetime.now(timezone.utc).isoformat()
        provenance='LIVE USGS, retrieved '+retrieved_at
    except Exception as error:
        data=load('seismic_sample.geojson')
        print('Live request unavailable:',type(error).__name__,str(error)[:180])
print(provenance)
events=normalize_quakes(data)
print(f'{len(events)} valid events; {sum(e["magnitude"] is None for e in events)} missing magnitudes')
display(events[:5])

In [ ]:
known=[e for e in events if e['magnitude'] is not None]
fig,ax=plt.subplots()
ax.scatter([e['magnitude'] for e in known],[e['depth_km'] for e in known],s=70,alpha=.75)
ax.invert_yaxis();ax.set(xlabel='Magnitude',ylabel='Depth (km, increasing downward)',title=provenance)
plt.show()

## Freshness has two clocks

Retrieval time records when you received a feed. Event time records when an event happened. A successful HTTP response can contain old observations. Inspect both; do not relabel old data as fresh because a page refreshed.

**Try:** set live mode; compare the largest event to the deepest. Disconnect the network and re-run to see the explicit fallback. **Checkpoint:** a missing magnitude stays `None`; magnitude zero remains zero.

In [ ]:
assert any(e['magnitude']==0 for e in normalize_quakes(load('seismic_sample.geojson')))
export('seismic-evidence.json',{'classification':provenance,'retrieved_at':retrieved_at,
    'source_url':USGS if retrieved_at else None,'events':events})

## Sources and attribution

[USGS GeoJSON summary format](https://earthquake.usgs.gov/earthquakes/feed/v1.0/geojson.php) defines coordinates, depth, event time, and feed metadata.

World View inspiration: [Kevin (Khoa) To, MIT](https://github.com/kevtoe/worldview). Original lab code, figures, and synthetic fixtures: JupyterLite WorldView contributors, MIT. See the [book references](https://jltobias.github.io/JupyterLite-WorldView/book/references.html) and repository notices for dependency and service terms.